# Automation Notebook

这个 notebook 是交互入口，复杂逻辑都在 `core/`。

建议按顺序执行：
1. 导入后台模块
2. 调整路径配置
3. 从 Aspen Excel 提取 JSON
4. 导出总表 CSV
5. 按现有模板填充四个 CAD CSV
6. 需要时执行 AutoCAD 回填

```[TODO] VR0118 质量有误不守恒，手动调整了输入 PET_Flow.xlsx```

In [ ]:
from core.paths import *
from core.stream_pipeline import (
    hack_stream,
    hack_stream_based_on,
    export_split_csv_files,
    export_full_csv,
    extract_streams,
    write_json,
    run_all,
)
from core.cad_applier import apply_to_autocad

print("[IN] Aspen 导出的流股数据位置", XLSX_PATH := DEFAULT_XLSX_PATH)
print("[IN] Aspen -> CAD 流股 ID 映射表位置", MAP_PATH := DEFAULT_MAP_PATH)
print("[OUT] 流股数据解析得到的 JSON 输出位置", JSON_PATH := DEFAULT_JSON_PATH)
print("[OUT] 根据 JSON 过滤重建的总 CSV 表格输出位置", FULL_CSV_PATH := DEFAULT_FULL_CSV_PATH)
print("[OUT] 根据 JSON 过滤重建的切分 CSV 表格输出位置", SPLIT_CSV_DIR := DEFAULT_SPLIT_CSV_DIR)

In [ ]:
json_data = extract_streams(
    xlsx_path=XLSX_PATH,
    output_path=JSON_PATH,
    # 默认取 Aspen 导出的流股数据的第一个工作表
    sheet_name=None,
    # hackdata=
)

In [ ]:
expanded_data = export_full_csv(
    input_path=JSON_PATH,
    map_path=MAP_PATH,
    output_path=FULL_CSV_PATH,
)

In [ ]:
cad_csv_files = export_split_csv_files(
    input_path=JSON_PATH,
    map_path=MAP_PATH,
    output_dir=SPLIT_CSV_DIR,
)

cad_csv_files

In [ ]:
try:
    # 只有在当前环境能连上 AutoCAD COM 时再执行
    apply_to_autocad(
        json_path=JSON_PATH,
        map_path=MAP_PATH,
    )
except RuntimeError as err:
    print("[Error]", err)
